# Portfolio allocation with HMM and semivariance

This walk-forward backtest estimates regimes from SPY volatility and semivariance and uses URTH as a benchmark. It can also use the VIX as a model input.

The notebook downloads daily split-adjusted prices from Yahoo Finance with `yfinance`. It saves its charts to `results/hmm-semivariance-allocation/`.


In [ ]:
"""
REGIME-CONDITIONAL PORTFOLIO STRATEGIES WITH HAR-SEMIVARIANCE AND HMM
======================================================================

WALK-FORWARD VERSION - Look-ahead bias eliminated, proper HRP & MST
WITH URTH BUY & HOLD BENCHMARK COMPARISON

This script implements a regime-based portfolio allocation system with:

REGIME DETECTION (Walk-Forward with HMM):
-----------------------------------------
- Expanding window Hidden Markov Model on HAR-semivariance features
- StandardScaler and GaussianHMM re-fit at each rebalancing point using only past data
- Regime labels assigned using only information available up to t-1
- HMM replaces K-Means for more sophisticated temporal regime modeling

PORTFOLIO STRATEGIES BY REGIME:
--------------------------------
1. LOW VOLATILITY: Maximum Sharpe Ratio (100% exposure)
2. NORMAL VOLATILITY: Hierarchical Risk Parity (canonical implementation)
3. HIGH VOLATILITY: MST Safe Haven with Mantegna distance

BENCHMARK COMPARISON:
---------------------
- URTH (iShares MSCI World ETF) Buy & Hold strategy
- Direct performance comparison with strategy equity curve
- Comparative metrics: returns, volatility, Sharpe, drawdown

NO LOOK-AHEAD BIAS - Walk-forward regime detection and feature engineering
"""

from pathlib import Path

import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import LedoitWolf
from scipy.optimize import minimize
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
import networkx as nx
import warnings

from _market_data import download_ohlcv
warnings.filterwarnings('ignore')

# Import HMM library
try:
    from hmmlearn import hmm
except ImportError:
    print("ERROR: hmmlearn not installed. Install with: pip install hmmlearn")
    raise

# ============================================================================
# CONFIGURATION
# ============================================================================

REGIME_TICKER = 'SPY'
BENCHMARK_TICKER = 'URTH'  # iShares MSCI World ETF for Buy & Hold comparison
START_DATE = '2013-01-01'
END_DATE = '2018-12-31'

GROWTH_ASSETS = ['SPY', 'QQQ', 'IWM', 'VTI', 'XLK', 'XLV', 'XLF', 'XLE', 'AAPL', 'MSFT']
DEFENSIVE_ASSETS = ['TLT', 'GLD', 'UUP', 'SH']
ALL_ASSETS = list(set(GROWTH_ASSETS + DEFENSIVE_ASSETS))

DAILY_LAG = 1
WEEKLY_LAG = 5
MONTHLY_LAG = 22
USE_VIX = True

N_REGIMES = 3
REGIME_REBALANCE_FREQ = 21
MIN_TRAIN_DAYS = 252  # Minimum training window for regime detection

# HMM-specific parameters
HMM_N_ITER = 100  # Maximum EM iterations
HMM_COVARIANCE_TYPE = 'full'  # Full covariance matrix for feature dependencies
HMM_RANDOM_STATE = 42

MAX_HRP_ASSETS = 10
TRANSACTION_COST = 0.001  # 10 basis points per trade
INITIAL_CAPITAL = 100000

# ============================================================================
# DATA ACQUISITION
# ============================================================================

def fetch_price_data(ticker, start_date, end_date):
    """Fetch daily split-adjusted OHLCV from Yahoo Finance."""
    columns = ["open", "high", "low", "close", "volume"]
    prices = download_ohlcv(ticker, start_date, end_date)
    return prices.rename(columns=str.lower)[columns]


def fetch_all_assets(tickers, start_date, end_date):
    """
    Fetch daily asset prices from Yahoo Finance.
    
    Returns dictionary: {ticker: DataFrame}
    Only includes tickers with sufficient data (>200 days).
    """
    print(f"\n{'='*80}")
    print("FETCHING ASSET DATA")
    print(f"{'='*80}")

    asset_data = {}
    for ticker in tickers:
        try:
            print(f"  Fetching {ticker}...", end='')
            df = fetch_price_data(ticker, start_date, end_date)
            if len(df) > 200:
                asset_data[ticker] = df
                print(f" ✓ {len(df)} days")
            else:
                print(f" ✗ Insufficient data")
        except Exception as e:
            print(f" ✗ Error: {str(e)}")

    print(f"\n✓ Successfully fetched {len(asset_data)} assets")
    return asset_data


# ============================================================================
# HAR-SEMIVARIANCE FEATURES (NO LOOK-AHEAD)
# ============================================================================

def calculate_garman_klass_vol(prices):
    """
    Calculate Garman-Klass volatility estimator.
    
    More efficient than close-to-close volatility as it uses OHLC data.
    Returns annualized volatility in percentage terms.
    
    Formula: sqrt(0.5 * (log(H/L))^2 - (2*log(2)-1) * (log(C/O))^2)
    """
    log_hl = np.log(prices['high'] / prices['low'])
    log_co = np.log(prices['close'] / prices['open'])
    gk_var = 0.5 * (log_hl ** 2) - (2 * np.log(2) - 1) * (log_co ** 2)
    
    # Annualize and convert to percentage
    rv_series = pd.Series(np.sqrt(gk_var.clip(lower=0) * 252) * 100, index=prices.index)
    return rv_series


def prepare_regime_features(spy_data, vix_data=None):
    """
    IMPROVED: Reduced feature set to avoid multicollinearity.
    
    Key changes:
    1. Use RV_D and RV spread (RV_D - RV_M) instead of all three
    2. Use VIX-RV differential instead of raw VIX levels
    3. Clip extreme values before creating features
    """
    print(f"\n{'='*80}")
    print("PREPARING REGIME DETECTION FEATURES (IMPROVED)")
    print(f"{'='*80}")

    df = pd.DataFrame(index=spy_data.index)

    # Calculate realized volatility
    rv = calculate_garman_klass_vol(spy_data)
    returns = np.log(spy_data['close'] / spy_data['close'].shift(1))

    # Semivariances
    gk_var = (rv / (np.sqrt(252) * 100)) ** 2
    rs_pos_array = np.where(returns > 0, gk_var, 0)
    rs_neg_array = np.where(returns <= 0, gk_var, 0)

    rs_pos = pd.Series(np.sqrt(rs_pos_array * 252) * 100, index=spy_data.index)
    rs_neg = pd.Series(np.sqrt(rs_neg_array * 252) * 100, index=spy_data.index)

    # IMPROVED FEATURE SET (less correlated, more robust)
    
    # Feature 1: Current realized vol (daily)
    df['RV_D'] = rv.shift(DAILY_LAG)
    
    # Feature 2: RV trend (spread between daily and monthly)
    # Positive = vol increasing, Negative = vol decreasing
    rv_monthly = rv.shift(1).rolling(MONTHLY_LAG).mean()
    df['RV_spread'] = df['RV_D'] - rv_monthly
    
    # Feature 3: Downside semivariance (daily only, no weekly)
    df['RS_neg_D'] = rs_neg.shift(DAILY_LAG)
    
    # Feature 4: Asymmetry ratio with robust calculation
    # Clip extreme values before division
    rs_pos_shifted = rs_pos.shift(DAILY_LAG).clip(lower=1.0)  # Minimum 1% to avoid division issues
    rs_neg_clipped = df['RS_neg_D'].clip(lower=0, upper=50)  # Cap at 50%
    df['Asym_ratio'] = rs_neg_clipped / rs_pos_shifted
    df['Asym_ratio'] = df['Asym_ratio'].clip(upper=10)  # Cap at 10x asymmetry
    
    # Feature 5: VIX differential (if available) - NOT raw VIX level
    if vix_data is not None:
        vix_aligned = vix_data.reindex(spy_data.index)
        vix_shifted = vix_aligned.shift(DAILY_LAG)
        
        # VIX - RV differential (fear premium)
        # Positive = implied vol > realized vol (fear)
        df['VIX_RV_diff'] = vix_shifted - df['RV_D']
    
    # Drop rows with NaN
    df = df.dropna()
    
    # NOTE: Winsorization removed from here to avoid look-ahead bias.
    # Percentile bounds must be computed on training data only, not the full sample.
    # Winsorization is now applied inside detect_regime_walk_forward_hmm()
    # using only data available up to the decision date.

    print(f"✓ Prepared {len(df)} observations with {len(df.columns)} features")
    print(f"  Features: {list(df.columns)}")
    print(f"  Feature correlations:")
    
    # Print correlation matrix to verify reduced multicollinearity
    corr = df.corr()
    for i, col1 in enumerate(df.columns):
        for col2 in df.columns[i+1:]:
            corr_val = corr.loc[col1, col2]
            if abs(corr_val) > 0.7:
                print(f"    WARNING: {col1} vs {col2}: {corr_val:.2f} (high correlation)")

    return df


def _forward_filter(model, X_scaled):
    """
    Forward-only HMM filter: compute P(state_t | x_1:t) at each time step.
    
    Unlike Viterbi (model.predict) or forward-backward (model.predict_proba),
    this uses ONLY past and present observations to determine the state at
    each time step. No future observations are used, eliminating look-ahead bias.
    
    Returns:
        filtered_states: array of most likely states at each time using only causal info
        log_alpha: forward log-probabilities (n_samples x n_states)
    """
    n_samples = X_scaled.shape[0]
    n_states = model.n_components
    
    # Log emission probabilities P(x_t | state_j)
    log_emission = model._compute_log_likelihood(X_scaled)
    
    log_startprob = np.log(model.startprob_ + 1e-300)
    log_transmat = np.log(model.transmat_ + 1e-300)
    
    # Forward pass in log-space
    log_alpha = np.zeros((n_samples, n_states))
    
    # t=0: P(state_0) * P(x_0 | state_0)
    log_alpha[0] = log_startprob + log_emission[0]
    
    for t in range(1, n_samples):
        for j in range(n_states):
            # sum_i [ alpha(t-1, i) * P(state_j | state_i) ] * P(x_t | state_j)
            log_alpha[t, j] = (
                np.logaddexp.reduce(log_alpha[t-1] + log_transmat[:, j])
                + log_emission[t, j]
            )
    
    # Most likely state at each time using only past observations
    filtered_states = np.argmax(log_alpha, axis=1)
    
    return filtered_states, log_alpha


def detect_regime_walk_forward_hmm(features, current_date, n_regimes=3, min_train_days=252):
    """
    Walk-forward HMM regime detection with forward-only filtering.
    
    FIX: Uses forward-only filter instead of Viterbi/forward-backward to
    eliminate look-ahead bias. Viterbi and forward-backward use the entire
    observation sequence (including future data) to determine each state,
    which leaks future information. The forward filter uses only past and
    present observations: P(state_t | x_1, ..., x_t).
    """
    train_mask = features.index < current_date
    train_data = features.loc[train_mask]

    if len(train_data) < min_train_days:
        return None, None, None

    X_train = train_data.copy()
    
    # FIX: Walk-forward winsorization using ONLY training data quantiles
    # This prevents look-ahead bias from full-sample percentile computation
    clip_bounds = {}
    for col in X_train.columns:
        p01 = X_train[col].quantile(0.01)
        p99 = X_train[col].quantile(0.99)
        clip_bounds[col] = (p01, p99)
        X_train[col] = X_train[col].clip(lower=p01, upper=p99)
    
    X_train_vals = X_train.values
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_vals)
    
    try:
        model = hmm.GaussianHMM(
            n_components=n_regimes,
            covariance_type='diag',
            n_iter=50,
            random_state=HMM_RANDOM_STATE,
            verbose=False,
            tol=1e-2,
            min_covar=1e-3
        )

        model.fit(X_train_scaled)
        
        if hasattr(model, 'monitor_') and not model.monitor_.converged:
            print(f"  WARNING: HMM did not fully converge at {current_date.date()}, using best fit")

        # FIX: Use forward-only filter instead of Viterbi (model.predict)
        # Forward filter computes P(state_t | x_1:t) using only causal data
        # Viterbi uses the FULL sequence (including future) to smooth states
        train_states, log_alpha_train = _forward_filter(model, X_train_scaled)

        # Map HMM states to regime labels by sorting on mean RV_D
        regime_means = []
        for i in range(n_regimes):
            state_mask = train_states == i
            if state_mask.sum() > 0:
                mean_rv = train_data.iloc[state_mask].loc[:, 'RV_D'].mean()
                regime_means.append((i, mean_rv))

        regime_means_sorted = sorted(regime_means, key=lambda x: x[1])
        state_to_regime = {old: new for new, (old, _) in enumerate(regime_means_sorted)}

        if current_date not in features.index:
            return None, scaler, model

        X_current = features.loc[current_date:current_date].copy()
        # Clip current observation using training-only bounds
        for col_idx, col in enumerate(features.columns):
            if col in clip_bounds:
                p01, p99 = clip_bounds[col]
                X_current.iloc[0, col_idx] = np.clip(X_current.iloc[0, col_idx], p01, p99)
        X_current_scaled = scaler.transform(X_current.values)

        # FIX: Forward-only prediction for current observation
        # Propagate the last filtered state through transition matrix,
        # then multiply by emission probability of current observation.
        # This uses only: past filtered state + transition model + current obs.
        last_log_alpha = log_alpha_train[-1]  # Filtered state at last training date
        log_transmat = np.log(model.transmat_ + 1e-300)
        
        # Predict: P(state_T | x_1:T-1) = sum_j P(state_T | state_j) * P(state_j | x_1:T-1)
        log_pred = np.array([
            np.logaddexp.reduce(last_log_alpha + log_transmat[:, j])
            for j in range(n_regimes)
        ])
        
        # Update with current emission: P(state_T | x_1:T) ∝ P(x_T | state_T) * P(state_T | x_1:T-1)
        log_emission_current = model._compute_log_likelihood(X_current_scaled)[0]
        log_posterior = log_pred + log_emission_current
        
        state_label = np.argmax(log_posterior)
        regime_label = state_to_regime[state_label]

        return regime_label, scaler, model

    except Exception as e:
        print(f"  HMM fitting error at {current_date.date()}: {str(e)}")
        return None, None, None


# ============================================================================
# CANONICAL HRP IMPLEMENTATION
# ============================================================================

def get_quasi_diag(link):
    """
    Reorder assets based on hierarchical tree clustering.
    
    This implements the quasi-diagonalization step of HRP.
    Returns list of asset indices in quasi-diagonal order.
    
    The quasi-diagonal ordering places similar assets close together,
    which is essential for the recursive bisection step.
    """
    link = link.astype(int)
    sort_ix = pd.Series([link[-1, 0], link[-1, 1]])
    num_items = link[-1, 3]

    while sort_ix.max() >= num_items:
        sort_ix.index = range(0, sort_ix.shape[0] * 2, 2)
        df0 = sort_ix[sort_ix >= num_items]
        i = df0.index
        j = df0.values - num_items
        sort_ix[i] = link[j, 0]
        df0 = pd.Series(link[j, 1], index=i + 1)
        sort_ix = pd.concat([sort_ix, df0])
        sort_ix = sort_ix.sort_index()
        sort_ix.index = range(sort_ix.shape[0])

    return sort_ix.tolist()


def get_cluster_var(cov, c_items):
    """
    Compute variance of a cluster using inverse-variance weights.
    
    This is used in the recursive bisection step to determine
    how to allocate weight between two clusters.
    
    Lower cluster variance -> higher weight allocation.
    """
    cov_slice = cov.loc[c_items, c_items]
    inv_diag = 1 / np.diag(cov_slice)
    w = inv_diag / inv_diag.sum()
    c_var = np.dot(w, np.dot(cov_slice, w))
    return c_var


def get_rec_bipart(cov, sort_ix):
    """
    Recursive bisection to compute HRP weights.

    This is the CANONICAL HRP algorithm from Lopez de Prado:
    1. Start with all assets, each with weight = 1
    2. Recursively bisect clusters into two sub-clusters
    3. Allocate weight between sub-clusters based on inverse cluster variance
    4. Continue until each cluster contains exactly one asset
    
    The result is a diversified portfolio that respects the hierarchical
    structure of asset correlations.
    """
    w = pd.Series(1.0, index=sort_ix)
    c_items = [sort_ix]

    # Recursive bisection loop
    while len(c_items) > 0:
        # Split each cluster into two halves
        c_items = [i[j:k] for i in c_items 
                   for j, k in ((0, len(i) // 2), (len(i) // 2, len(i))) 
                   if len(i) > 1]

        # Allocate weights between each pair of sub-clusters
        for i in range(0, len(c_items), 2):
            c_items0 = c_items[i]      # Left sub-cluster
            c_items1 = c_items[i + 1]  # Right sub-cluster

            # Compute cluster variances
            c_var0 = get_cluster_var(cov, c_items0)
            c_var1 = get_cluster_var(cov, c_items1)

            # Allocate weight: inverse variance weighting
            alpha = 1 - c_var0 / (c_var0 + c_var1)

            # Scale weights for both sub-clusters
            w[c_items0] *= alpha       # Left cluster gets alpha
            w[c_items1] *= 1 - alpha   # Right cluster gets 1-alpha

    return w


def hrp_portfolio_canonical(returns, max_assets=None):
    """
    Canonical Hierarchical Risk Parity portfolio implementation.

    Three-step process:
    1. Tree Clustering: Build hierarchical tree using correlation distance
    2. Quasi-Diagonalization: Reorder assets based on tree structure
    3. Recursive Bisection: Allocate weights using inverse cluster variance
    
    This implementation follows Lopez de Prado's original HRP algorithm.
    
    Parameters:
    - returns: DataFrame of asset returns
    - max_assets: Optional limit on number of assets (selects top N by weight)
    
    Returns dict: {asset: weight}
    """
    # Need sufficient data for stable covariance estimation
    if len(returns) < 60:
        return {}

    try:
        # Step 1: Hierarchical Tree Clustering
        # Use correlation distance: d = sqrt((1 - corr) / 2)
        corr = returns.corr()
        dist = np.sqrt((1 - corr) / 2)
        
        # Convert to condensed distance matrix for scipy
        dist_condensed = squareform(dist.values, checks=False)
        
        # Single linkage clustering (as per original HRP)
        link = linkage(dist_condensed, method='single')

        # Step 2: Quasi-Diagonalization
        sort_ix = get_quasi_diag(link)
        sort_ix = [returns.columns[i] for i in sort_ix]

        # Step 3: Recursive Bisection
        # Use annualized covariance matrix
        cov = returns.cov() * 252
        weights = get_rec_bipart(cov, sort_ix)

        # Optional: Limit to top N assets by weight to reduce turnover
        if max_assets is not None and len(weights) > max_assets:
            weights = weights.nlargest(max_assets)
            weights = weights / weights.sum()

        # Filter out negligible weights
        return {asset: w for asset, w in weights.items() if w > 0.01}

    except Exception as e:
        print(f"  HRP error: {str(e)}")
        return {}


# ============================================================================
# MST PORTFOLIO WITH MANTEGNA DISTANCE
# ============================================================================

def mst_safe_haven_portfolio_mantegna(returns, defensive_assets):
    """
    MST Safe Haven portfolio using Mantegna distance metric.

    Strategy:
    1. Filter to defensive assets only (bonds, gold, etc.)
    2. Compute Mantegna distance: d_ij = sqrt(2 * (1 - rho_ij))
    3. Build Minimum Spanning Tree (MST) to identify core defensive assets
    4. Weight MST assets using inverse volatility
    
    Mantegna distance is a proper metric that satisfies triangle inequality,
    making it suitable for graph-based portfolio construction.
    
    The MST identifies the most diverse subset of defensive assets,
    providing robust protection during high volatility regimes.
    
    Parameters:
    - returns: DataFrame of all asset returns
    - defensive_assets: List of defensive asset tickers
    
    Returns dict: {asset: weight}
    """
    # Filter to available defensive assets
    available_defensive = [a for a in defensive_assets if a in returns.columns]

    if len(available_defensive) < 2:
        # Not enough defensive assets - equal weight
        weights = {asset: 1.0 / len(available_defensive) for asset in available_defensive}
        return weights

    returns_defensive = returns[available_defensive]

    # Need sufficient data for stable correlation estimation
    if len(returns_defensive) < 60:
        weights = {asset: 1.0 / len(available_defensive) for asset in available_defensive}
        return weights

    try:
        # Use Spearman correlation for robustness to outliers and non-linearity
        corr_matrix = returns_defensive.corr(method='spearman')

        # Build fully connected graph with Mantegna distances
        G = nx.Graph()
        for i, asset_i in enumerate(available_defensive):
            for j, asset_j in enumerate(available_defensive):
                if i < j:
                    rho = corr_matrix.iloc[i, j]
                    # Mantegna distance metric
                    distance = np.sqrt(2 * (1 - rho))
                    G.add_edge(asset_i, asset_j, weight=distance)

        # Compute Minimum Spanning Tree
        # MST selects most diverse subset of assets (maximizes distance)
        mst = nx.minimum_spanning_tree(G)

        # Weight MST assets using inverse volatility
        # Lower volatility -> higher weight (conservative approach for safe haven)
        vols = returns_defensive.std() * np.sqrt(252)
        inv_vols = 1 / vols

        mst_assets = list(mst.nodes())
        inv_vols_mst = inv_vols[mst_assets]
        weights_mst = inv_vols_mst / inv_vols_mst.sum()

        # Filter out negligible weights
        return {asset: w for asset, w in weights_mst.items() if w > 0.01}

    except Exception as e:
        print(f"  MST error: {str(e)}")
        # Fallback to equal weight
        weights = {asset: 1.0 / len(available_defensive) for asset in available_defensive}
        return weights


# ============================================================================
# PORTFOLIO OPTIMIZATION METHODS
# ============================================================================

def max_sharpe_portfolio(returns, risk_free_rate=0.02):
    """
    Maximum Sharpe Ratio portfolio optimization.
    
    Used in LOW VOLATILITY regime to maximize risk-adjusted returns.
    
    Optimization problem:
    - Maximize: (portfolio_return - risk_free_rate) / portfolio_volatility
    - Subject to: sum(weights) = 1, weights >= 0, individual weight <= 30%
    
    Uses Ledoit-Wolf shrinkage for robust covariance estimation.
    
    Parameters:
    - returns: DataFrame of asset returns
    - risk_free_rate: Annual risk-free rate (default 2%)
    
    Returns dict: {asset: weight}
    """
    # Need sufficient data for stable optimization
    if len(returns) < 60:
        return {}

    # Compute expected returns (annualized mean)
    mu = returns.mean() * 252

    try:
        # Use Ledoit-Wolf shrinkage for robust covariance estimation
        # This reduces estimation error in covariance matrix
        lw = LedoitWolf()
        cov_matrix = lw.fit(returns.values).covariance_ * 252
    except:
        return {}

    n_assets = len(mu)

    # Objective: negative Sharpe ratio (we minimize)
    def neg_sharpe(weights):
        port_return = np.dot(weights, mu)
        port_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
        return -(port_return - risk_free_rate) / port_vol if port_vol > 0 else -np.inf

    # Constraints: weights sum to 1
    constraints = [{'type': 'eq', 'fun': lambda x: np.sum(x) - 1.0}]
    
    # Bounds: long only, max 30% per asset to ensure diversification
    bounds = [(0, 0.3) for _ in range(n_assets)]
    
    # Initial guess: equal weight
    x0 = np.array([1.0 / n_assets] * n_assets)

    # Run optimization using SLSQP (Sequential Least Squares Programming)
    result = minimize(neg_sharpe, x0, method='SLSQP', bounds=bounds, 
                     constraints=constraints, options={'maxiter': 1000})

    if result.success:
        weights = result.x
        # Filter out negligible weights
        weights = np.where(weights < 0.01, 0, weights)
        if np.sum(weights) > 0:
            weights = weights / np.sum(weights)
            return {asset: w for asset, w in zip(returns.columns, weights) if w > 0.01}

    return {}


# ============================================================================
# BENCHMARK: BUY & HOLD STRATEGY
# ============================================================================

def create_buy_hold_benchmark(benchmark_data, regime_dates, initial_capital, transaction_cost):
    """
    Create Buy & Hold benchmark strategy for comparison.
    
    Strategy:
    - Buy at first available date
    - Hold until end (no rebalancing)
    - Account for transaction costs on initial purchase
    
    This represents a passive investment approach for comparison with
    the active regime-conditional strategy.
    
    Parameters:
    - benchmark_data: DataFrame with OHLC data for benchmark ticker
    - regime_dates: List of dates to align with strategy
    - initial_capital: Starting capital
    - transaction_cost: Transaction cost rate
    
    Returns DataFrame with columns: date, value, return
    """
    print(f"\n{'='*80}")
    print("CREATING BUY & HOLD BENCHMARK")
    print(f"{'='*80}")
    
    # Align benchmark data to regime dates
    benchmark_aligned = benchmark_data.reindex(regime_dates)
    
    # Find first valid date with price data
    first_valid_idx = benchmark_aligned['close'].first_valid_index()
    if first_valid_idx is None:
        print("  ✗ No valid benchmark data")
        return None
    
    first_price = benchmark_aligned.loc[first_valid_idx, 'close']
    
    # Account for transaction cost on initial purchase
    net_capital = initial_capital * (1 - transaction_cost)
    shares = net_capital / first_price
    
    print(f"  Initial purchase: {shares:.2f} shares at ${first_price:.2f}")
    print(f"  Transaction cost: ${initial_capital * transaction_cost:.2f}")
    
    # Calculate portfolio value for each date
    benchmark_values = []
    for date in regime_dates:
        if date >= first_valid_idx and not pd.isna(benchmark_aligned.loc[date, 'close']):
            current_price = benchmark_aligned.loc[date, 'close']
            portfolio_value = shares * current_price
            
            benchmark_values.append({
                'date': date,
                'value': portfolio_value,
                'price': current_price
            })
    
    benchmark_df = pd.DataFrame(benchmark_values)
    benchmark_df['return'] = benchmark_df['value'].pct_change()
    
    print(f"  ✓ Benchmark created: {len(benchmark_df)} days")
    print(f"  Final value: ${benchmark_df['value'].iloc[-1]:,.2f}")
    
    return benchmark_df


# ============================================================================
# BACKTEST ENGINE WITH WALK-FORWARD HMM REGIME DETECTION
# ============================================================================

class RegimePortfolioBacktestWalkForwardHMM:
    """
    Backtest engine with walk-forward HMM regime detection.
    
    NO LOOK-AHEAD BIAS:
    - Regime detection uses only data up to t-1
    - Portfolio optimization uses only data up to t-1
    - Execution at t uses open price (known at market open)
    - Portfolio valuation at t uses close price
    
    Trading Flow:
    Day T-1 (Close):
    - Portfolio valued at Close[T-1]
    - Features from T-2 data
    - Regime detected using data up to T-1
    - Optimization using returns up to T-1
    - Signals generated
    
    Day T (Open):
    - Execute pending signals at Open[T]
    - Portfolio positions updated
    
    Day T (Close):
    - Portfolio valued at Close[T]
    - Returns: Close[T]/Close[T-1] - 1
    """

    def __init__(self, regime_features, asset_data, initial_capital, transaction_cost):
        self.regime_features = regime_features
        self.asset_data = asset_data
        self.initial_capital = initial_capital
        self.transaction_cost = transaction_cost

        self.capital = initial_capital
        self.cash = initial_capital
        self.positions = {}  # {ticker: shares}
        self.portfolio_history = []
        self.trades = []
        self.regime_history = []

        self.align_asset_data()

    def align_asset_data(self):
        """Align all asset data to common regime feature dates."""
        common_dates = self.regime_features.index
        for ticker in self.asset_data:
            self.asset_data[ticker] = self.asset_data[ticker].reindex(common_dates)

    def get_current_prices(self, date, price_type='close'):
        """
        Get current prices for all assets at given date.
        
        Parameters:
        - date: Date to get prices for
        - price_type: 'open' or 'close'
        
        Returns dict: {ticker: price}
        """
        prices = {}
        for ticker, df in self.asset_data.items():
            if date in df.index and not pd.isna(df.loc[date, price_type]):
                prices[ticker] = df.loc[date, price_type]
        return prices

    def calculate_portfolio_value(self, date):
        """
        Calculate total portfolio value at given date.
        
        Portfolio value = cash + sum(shares * price)
        Uses close prices for valuation.
        """
        prices = self.get_current_prices(date, 'close')
        risky_value = sum(self.positions.get(ticker, 0) * prices.get(ticker, 0) 
                         for ticker in self.positions)
        return self.cash + risky_value

    def execute_rebalance(self, date, target_weights):
        """
        Execute portfolio rebalancing with realistic timing.
        
        IMPORTANT: To avoid look-ahead bias:
        - Decision made at date (using close price for valuation)
        - Execution at next date's open price
        - Transaction costs applied to all trades
        
        Parameters:
        - date: Current date (decision date)
        - target_weights: Dict {ticker: weight} representing target allocation
        """
        dates_list = list(self.regime_features.index)
        current_idx = dates_list.index(date)

        # Need next day for execution
        if current_idx >= len(dates_list) - 1:
            return

        next_date = dates_list[current_idx + 1]
        
        # Execute at next day's OPEN price (realistic execution)
        exec_prices = self.get_current_prices(next_date, 'open')

        if not exec_prices:
            return

        # Calculate portfolio value at current date's CLOSE
        # This is the capital available for rebalancing
        portfolio_value = self.calculate_portfolio_value(date)

        # Calculate target positions in shares
        target_positions = {}
        for ticker, weight in target_weights.items():
            if ticker in exec_prices:
                target_value = portfolio_value * weight
                target_positions[ticker] = target_value / exec_prices[ticker]

        # Step 1: Close positions not in target (sell unwanted holdings)
        for ticker in list(self.positions.keys()):
            if ticker not in target_positions and self.positions[ticker] != 0:
                if ticker in exec_prices:
                    sell_value = self.positions[ticker] * exec_prices[ticker]
                    cost = abs(sell_value) * self.transaction_cost
                    self.cash += sell_value - cost
                    self.positions[ticker] = 0

                    self.trades.append({
                        'date': next_date,
                        'ticker': ticker,
                        'action': 'SELL',
                        'value': sell_value,
                        'cost': cost
                    })

        # Step 2: Adjust to target positions (buy/sell to reach targets)
        for ticker, target_shares in target_positions.items():
            current_shares = self.positions.get(ticker, 0)
            delta_shares = target_shares - current_shares

            # Only trade if delta is significant (avoid tiny trades)
            if abs(delta_shares) > 0.01:
                trade_value = delta_shares * exec_prices[ticker]
                cost = abs(trade_value) * self.transaction_cost

                if trade_value > 0:  # BUY
                    # Check if we have enough cash
                    if self.cash >= (trade_value + cost):
                        self.positions[ticker] = target_shares
                        self.cash -= (trade_value + cost)

                        self.trades.append({
                            'date': next_date,
                            'ticker': ticker,
                            'action': 'BUY',
                            'value': trade_value,
                            'cost': cost
                        })

                else:  # SELL
                    self.positions[ticker] = target_shares
                    self.cash += (-trade_value - cost)

                    self.trades.append({
                        'date': next_date,
                        'ticker': ticker,
                        'action': 'SELL',
                        'value': trade_value,
                        'cost': cost
                    })

    def run_backtest(self):
        print(f"\n{'='*80}")
        print("RUNNING WALK-FORWARD HMM REGIME-CONDITIONAL BACKTEST")
        print(f"{'='*80}")

        dates = list(self.regime_features.index)
        last_rebalance = -REGIME_REBALANCE_FREQ

        for day_num, date in enumerate(dates):
            # CRITICAL FIX: Clarify we're at END of previous day making decisions
            # for execution at OPEN of current day
            
            # Skip first day (need previous day for valuation)
            if day_num == 0:
                portfolio_value = self.initial_capital
                self.portfolio_history.append({
                    'date': date,
                    'value': portfolio_value,
                    'cash': self.cash,
                    'regime': None,
                    'regime_name': 'Initialization',
                    'num_positions': 0
                })
                continue

            # ====================================================================
            # DECISION MADE AT END OF PREVIOUS DAY (date - 1)
            # ====================================================================
            prev_date = dates[day_num - 1]
            
            # REGIME DETECTION: Use data up to (and including) prev_date
            # Features for prev_date use data up to prev_date-1 (already lagged)
            regime_label, scaler, hmm_model = detect_regime_walk_forward_hmm(
                self.regime_features, prev_date, N_REGIMES, MIN_TRAIN_DAYS
            )

            if regime_label is None:
                # Not enough training data yet - stay in cash
                # FIX: Value portfolio at previous close
                portfolio_value = self.calculate_portfolio_value(prev_date)
                self.portfolio_history.append({
                    'date': date,
                    'value': portfolio_value,
                    'cash': self.cash,
                    'regime': None,
                    'regime_name': 'Insufficient Data',
                    'num_positions': len([p for p in self.positions.values() if p != 0])
                })
                continue

            regime_name = ['Low Volatility', 'Normal Volatility', 'High Volatility'][regime_label]

            self.regime_history.append({
                'date': prev_date,  # Regime known at end of prev_date
                'regime': regime_label,
                'regime_name': regime_name
            })

            # FIX: Portfolio value at END of previous day (Close[T-1])
            # This is the capital available for today's trades
            portfolio_value = self.calculate_portfolio_value(prev_date)

            # REBALANCING LOGIC
            days_since_rebal = day_num - last_rebalance

            if days_since_rebal >= REGIME_REBALANCE_FREQ and day_num >= MIN_TRAIN_DAYS + 60:
                # Historical returns for optimization: up to (and including) prev_date
                # Returns calculated from close prices, so last return is:
                # Close[prev_date]/Close[prev_date-1] - 1
                hist_start = max(0, day_num - 252)
                hist_end = day_num  # Python slicing: [hist_start:hist_end] excludes hist_end
                hist_dates = dates[hist_start:hist_end]  # Up to prev_date

                returns_list = []
                for ticker in self.asset_data:
                    ticker_prices = self.asset_data[ticker].loc[hist_dates, 'close']
                    ticker_returns = ticker_prices.pct_change().dropna()
                    returns_list.append(ticker_returns.rename(ticker))

                if returns_list:
                    returns_df = pd.concat(returns_list, axis=1).dropna()

                    # REGIME-CONDITIONAL ALLOCATION
                    if regime_label == 0:
                        growth_returns = returns_df[[c for c in GROWTH_ASSETS if c in returns_df.columns]]
                        target_weights = max_sharpe_portfolio(growth_returns)
                        strategy_name = "Max Sharpe"

                    elif regime_label == 1:
                        all_returns = returns_df[[c for c in GROWTH_ASSETS + DEFENSIVE_ASSETS if c in returns_df.columns]]
                        target_weights = hrp_portfolio_canonical(all_returns, max_assets=MAX_HRP_ASSETS)
                        strategy_name = "HRP (Canonical)"

                    else:
                        all_returns = returns_df
                        target_weights = mst_safe_haven_portfolio_mantegna(all_returns, DEFENSIVE_ASSETS)
                        strategy_name = "MST Safe Haven (Mantegna)"

                    if target_weights:
                        print(f"\n  Day {day_num}: {prev_date.strftime('%Y-%m-%d')} (Decision) -> {date.strftime('%Y-%m-%d')} (Execution)")
                        print(f"    Regime: {regime_name}")
                        print(f"    Strategy: {strategy_name}")
                        print(f"    Assets: {list(target_weights.keys())}")

                        # FIX: Pass prev_date as decision date
                        # execute_rebalance will execute at OPEN of next day (date)
                        self.execute_rebalance(prev_date, target_weights)
                        last_rebalance = day_num

            # ====================================================================
            # EXECUTION AT OPEN[T] - positions already updated by execute_rebalance
            # VALUATION AT CLOSE[T] - for performance reporting
            # ====================================================================
            current_value = self.calculate_portfolio_value(date)

            self.portfolio_history.append({
                'date': date,
                'value': current_value,
                'cash': self.cash,
                'regime': regime_label,
                'regime_name': regime_name,
                'num_positions': len([p for p in self.positions.values() if p != 0])
            })

            if day_num % 100 == 0:
                print(f"  Progress: {day_num}/{len(dates)} days")

        print(f"\n✓ Backtest complete (Walk-Forward HMM - NO LOOK-AHEAD)")
        return self.portfolio_history, self.trades


# ============================================================================
# PERFORMANCE METRICS
# ============================================================================

def calculate_performance_metrics(portfolio_df, initial_capital):
    """
    Calculate comprehensive performance metrics.
    
    Metrics:
    - Total Return (%)
    - Annualized Return (%)
    - Annualized Volatility (%)
    - Sharpe Ratio (assuming 2% risk-free rate)
    - Maximum Drawdown (%)
    - Calmar Ratio (Return/MaxDD)
    - Win Rate (% of positive return days)
    
    Returns dict of metrics.
    """
    final_value = portfolio_df['value'].iloc[-1]
    total_return = (final_value / initial_capital - 1) * 100

    returns = portfolio_df['value'].pct_change().dropna()
    
    # Annualized metrics (assuming 252 trading days per year)
    annual_return = returns.mean() * 252 * 100
    annual_vol = returns.std() * np.sqrt(252) * 100
    sharpe = (annual_return - 2.0) / annual_vol if annual_vol > 0 else 0

    # Drawdown analysis
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_dd = drawdown.min() * 100

    # Calmar ratio: return / max drawdown
    calmar = annual_return / abs(max_dd) if max_dd != 0 else 0

    # Win rate
    win_rate = (returns > 0).sum() / len(returns) * 100 if len(returns) > 0 else 0

    return {
        'total_return': total_return,
        'annual_return': annual_return,
        'annual_vol': annual_vol,
        'sharpe': sharpe,
        'max_dd': max_dd,
        'calmar': calmar,
        'win_rate': win_rate,
        'final_value': final_value
    }


def print_comparative_metrics(strategy_metrics, benchmark_metrics):
    """
    Print side-by-side comparison of strategy vs benchmark metrics.
    
    Formats metrics in a readable table for easy comparison.
    """
    print(f"\n{'='*80}")
    print("COMPARATIVE PERFORMANCE: STRATEGY VS BENCHMARK")
    print(f"{'='*80}")
    print(f"\n{'Metric':<25} {'Strategy':>15} {'Benchmark':>15} {'Difference':>15}")
    print("-" * 80)
    
    metrics_to_compare = [
        ('Final Value', 'final_value', '${:,.0f}'),
        ('Total Return', 'total_return', '{:.2f}%'),
        ('Annual Return', 'annual_return', '{:.2f}%'),
        ('Annual Volatility', 'annual_vol', '{:.2f}%'),
        ('Sharpe Ratio', 'sharpe', '{:.3f}'),
        ('Maximum Drawdown', 'max_dd', '{:.2f}%'),
        ('Calmar Ratio', 'calmar', '{:.3f}'),
        ('Win Rate', 'win_rate', '{:.2f}%')
    ]
    
    for label, key, fmt in metrics_to_compare:
        strat_val = strategy_metrics[key]
        bench_val = benchmark_metrics[key]
        
        # Calculate difference (absolute or percentage points)
        if key == 'final_value':
            diff = strat_val - bench_val
            diff_str = f"${diff:,.0f}"
        else:
            diff = strat_val - bench_val
            diff_str = f"{diff:+.2f}"
        
        print(f"{label:<25} {fmt.format(strat_val):>15} {fmt.format(bench_val):>15} {diff_str:>15}")
    
    print("-" * 80)


# ============================================================================
# VISUALIZATION WITH BENCHMARK
# ============================================================================

def plot_regime_portfolio_with_benchmark(portfolio_history, benchmark_df):
    """
    Visualize regime-conditional portfolio performance with benchmark comparison.
    
    Creates 5 subplots:
    1. Portfolio Value: Strategy vs Benchmark equity curves
    2. Regime Sequence: Timeline of detected regimes
    3. Cumulative Returns: Strategy vs Benchmark
    4. Rolling Sharpe Ratio: 63-day window (quarterly)
    5. Drawdown: Strategy vs Benchmark
    """

    portfolio_df = pd.DataFrame(portfolio_history)
    portfolio_df = portfolio_df[portfolio_df['regime'].notna()]

    fig, axes = plt.subplots(5, 1, figsize=(16, 18))

    regime_colors = {0: '#90EE90', 1: '#FFD700', 2: '#FF6B6B'}

    # ========================================================================
    # PLOT 1: PORTFOLIO VALUE - STRATEGY VS BENCHMARK
    # ========================================================================
    
    # Strategy equity curve with regime coloring
    for regime in [0, 1, 2]:
        mask = portfolio_df['regime'] == regime
        if mask.sum() > 0:
            axes[0].scatter(portfolio_df.loc[mask, 'date'],
                           portfolio_df.loc[mask, 'value'],
                           c=regime_colors[regime],
                           s=10,
                           alpha=0.6,
                           label=f"Strategy - {portfolio_df.loc[mask, 'regime_name'].iloc[0]}")

    # Benchmark equity curve
    axes[0].plot(benchmark_df['date'], benchmark_df['value'],
                color='black', linewidth=2, linestyle='--',
                label='Benchmark (URTH B&H)', alpha=0.7)

    axes[0].set_ylabel('Portfolio Value ($)', fontsize=11)
    axes[0].set_title('Strategy vs Benchmark: Equity Curves (Walk-Forward HMM Regime Detection)', 
                     fontsize=14, fontweight='bold')
    axes[0].legend(loc='upper left', fontsize=9)
    axes[0].grid(alpha=0.3)

    # ========================================================================
    # PLOT 2: REGIME SEQUENCE
    # ========================================================================
    
    axes[1].scatter(portfolio_df['date'],
                   portfolio_df['regime'],
                   c=[regime_colors[r] for r in portfolio_df['regime']],
                   s=20,
                   alpha=0.6)
    axes[1].set_ylabel('Regime', fontsize=11)
    axes[1].set_yticks([0, 1, 2])
    axes[1].set_yticklabels(['Low Vol\n(Max Sharpe)', 'Normal Vol\n(HRP)', 'High Vol\n(MST+SH)'])
    axes[1].set_title('Active Regime & Strategy (Walk-Forward HMM Detection)', fontsize=14, fontweight='bold')
    axes[1].grid(alpha=0.3)

    # ========================================================================
    # PLOT 3: CUMULATIVE RETURNS
    # ========================================================================
    
    # Strategy cumulative returns
    portfolio_df['return'] = portfolio_df['value'].pct_change()
    strategy_cum_ret = portfolio_df['return'].cumsum() * 100
    
    # Benchmark cumulative returns
    benchmark_cum_ret = benchmark_df['return'].cumsum() * 100
    
    axes[2].plot(portfolio_df['date'], strategy_cum_ret, 
                color='blue', linewidth=2, label='Strategy', alpha=0.8)
    axes[2].plot(benchmark_df['date'], benchmark_cum_ret,
                color='black', linewidth=2, linestyle='--', label='Benchmark (URTH)', alpha=0.7)
    axes[2].set_ylabel('Cumulative Return (%)', fontsize=11)
    axes[2].set_title('Cumulative Returns: Strategy vs Benchmark', fontsize=14, fontweight='bold')
    axes[2].legend()
    axes[2].grid(alpha=0.3)
    axes[2].axhline(y=0, color='black', linestyle='--', linewidth=1)

    # ========================================================================
    # PLOT 4: ROLLING SHARPE RATIO (63-day window = ~1 quarter)
    # ========================================================================
    
    # Strategy rolling Sharpe
    strategy_rolling_sharpe = (
        portfolio_df['return'].rolling(63).mean() * 252 / 
        (portfolio_df['return'].rolling(63).std() * np.sqrt(252))
    )
    
    # Benchmark rolling Sharpe
    benchmark_rolling_sharpe = (
        benchmark_df['return'].rolling(63).mean() * 252 / 
        (benchmark_df['return'].rolling(63).std() * np.sqrt(252))
    )
    
    axes[3].plot(portfolio_df['date'], strategy_rolling_sharpe,
                color='blue', linewidth=1.5, label='Strategy', alpha=0.8)
    axes[3].plot(benchmark_df['date'], benchmark_rolling_sharpe,
                color='black', linewidth=1.5, linestyle='--', label='Benchmark (URTH)', alpha=0.7)
    axes[3].set_ylabel('Rolling Sharpe (63d)', fontsize=11)
    axes[3].set_title('Rolling Sharpe Ratio: Strategy vs Benchmark', fontsize=14, fontweight='bold')
    axes[3].axhline(y=0, color='red', linestyle='--', linewidth=1)
    axes[3].legend()
    axes[3].grid(alpha=0.3)

    # ========================================================================
    # PLOT 5: DRAWDOWN COMPARISON
    # ========================================================================
    
    # Strategy drawdown
    strategy_returns = portfolio_df['return'].dropna()
    strategy_cumulative = (1 + strategy_returns).cumprod()
    strategy_running_max = strategy_cumulative.expanding().max()
    strategy_drawdown = (strategy_cumulative - strategy_running_max) / strategy_running_max

    # Benchmark drawdown
    benchmark_returns = benchmark_df['return'].dropna()
    benchmark_cumulative = (1 + benchmark_returns).cumprod()
    benchmark_running_max = benchmark_cumulative.expanding().max()
    benchmark_drawdown = (benchmark_cumulative - benchmark_running_max) / benchmark_running_max

    axes[4].fill_between(portfolio_df['date'].iloc[1:], strategy_drawdown.values * 100, 0,
                        color='blue', alpha=0.5, label='Strategy')
    axes[4].fill_between(benchmark_df['date'].iloc[1:], benchmark_drawdown.values * 100, 0,
                        color='black', alpha=0.3, label='Benchmark (URTH)')
    axes[4].set_ylabel('Drawdown (%)', fontsize=11)
    axes[4].set_xlabel('Date', fontsize=11)
    axes[4].set_title('Drawdown Comparison: Strategy vs Benchmark', fontsize=14, fontweight='bold')
    axes[4].legend()
    axes[4].grid(alpha=0.3)

    plt.tight_layout()
    Path("results/hmm-semivariance-allocation").mkdir(parents=True, exist_ok=True)
    plt.savefig("results/hmm-semivariance-allocation/regime_conditional_portfolio_hmm_with_benchmark.png", dpi=300, bbox_inches='tight')
    print(f"\n✓ Visualization saved as 'regime_conditional_portfolio_hmm_with_benchmark.png'")
    plt.show()


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """
    Execute regime-conditional portfolio system with walk-forward HMM analysis.
    
    Main execution flow:
    1. Fetch data (SPY for regimes, VIX for volatility, all assets, URTH for benchmark)
    2. Prepare HAR-semivariance features
    3. Run walk-forward backtest with HMM regime detection
    4. Create Buy & Hold benchmark
    5. Calculate and compare metrics
    6. Visualize results
    """

    print("="*80)
    print("REGIME-CONDITIONAL PORTFOLIO WITH HMM (NO LOOK-AHEAD)")
    print("WITH URTH BUY & HOLD BENCHMARK COMPARISON")
    print("="*80)

    try:
        # ====================================================================
        # DATA ACQUISITION
        # ====================================================================
        
        print(f"\nFetching {REGIME_TICKER} for regime detection...")
        spy_data = fetch_price_data(REGIME_TICKER, START_DATE, END_DATE)

        vix_data = None
        if USE_VIX:
            print("Fetching VIX...")
            try:
                vix_data = fetch_price_data('^VIX', START_DATE, END_DATE)['close']
            except RuntimeError as error:
                print(f"  {error}. Proceeding without VIX.")

        print(f"\nFetching {BENCHMARK_TICKER} for benchmark...")
        try:
            benchmark_data = fetch_price_data(BENCHMARK_TICKER, START_DATE, END_DATE)
        except RuntimeError as error:
            print(f"  {error}. Proceeding without a benchmark.")
            benchmark_data = None

        asset_data = fetch_all_assets(ALL_ASSETS, START_DATE, END_DATE)

        # ====================================================================
        # FEATURE PREPARATION
        # ====================================================================
        
        regime_features = prepare_regime_features(spy_data, vix_data)

        # ====================================================================
        # STRATEGY BACKTEST
        # ====================================================================
        
        backtest = RegimePortfolioBacktestWalkForwardHMM(
            regime_features=regime_features,
            asset_data=asset_data,
            initial_capital=INITIAL_CAPITAL,
            transaction_cost=TRANSACTION_COST
        )

        portfolio_history, trades = backtest.run_backtest()

        # ====================================================================
        # BENCHMARK CREATION
        # ====================================================================
        
        portfolio_df = pd.DataFrame(portfolio_history)
        portfolio_df = portfolio_df[portfolio_df['regime'].notna()]
        
        benchmark_df = None
        if benchmark_data is not None:
            benchmark_df = create_buy_hold_benchmark(
                benchmark_data=benchmark_data,
                regime_dates=portfolio_df['date'].tolist(),
                initial_capital=INITIAL_CAPITAL,
                transaction_cost=TRANSACTION_COST
            )

        # ====================================================================
        # PERFORMANCE METRICS
        # ====================================================================
        
        print(f"\n{'='*80}")
        print("STRATEGY PERFORMANCE SUMMARY")
        print(f"{'='*80}")

        # Calculate strategy metrics
        strategy_metrics = calculate_performance_metrics(portfolio_df, INITIAL_CAPITAL)
        
        print(f"\nStrategy Performance:")
        print(f"  Initial Capital:   ${INITIAL_CAPITAL:,.0f}")
        print(f"  Final Value:       ${strategy_metrics['final_value']:,.0f}")
        print(f"  Total Return:      {strategy_metrics['total_return']:.2f}%")
        print(f"  Annual Return:     {strategy_metrics['annual_return']:.2f}%")
        print(f"  Annual Volatility: {strategy_metrics['annual_vol']:.2f}%")
        print(f"  Sharpe Ratio:      {strategy_metrics['sharpe']:.3f}")
        print(f"  Maximum Drawdown:  {strategy_metrics['max_dd']:.2f}%")
        print(f"  Calmar Ratio:      {strategy_metrics['calmar']:.3f}")
        print(f"  Win Rate:          {strategy_metrics['win_rate']:.2f}%")
        print(f"  Total Trades:      {len(trades)}")

        # Calculate and compare benchmark metrics
        if benchmark_df is not None:
            benchmark_metrics = calculate_performance_metrics(benchmark_df, INITIAL_CAPITAL)
            print_comparative_metrics(strategy_metrics, benchmark_metrics)

        # ====================================================================
        # REGIME STATISTICS
        # ====================================================================
        
        print(f"\n{'='*80}")
        print("REGIME STATISTICS (Walk-Forward HMM Detection)")
        print(f"{'='*80}")

        for regime in [0, 1, 2]:
            regime_mask = portfolio_df['regime'] == regime
            if regime_mask.sum() > 0:
                regime_name = portfolio_df.loc[regime_mask, 'regime_name'].iloc[0]
                n_days = regime_mask.sum()
                pct = n_days / len(portfolio_df) * 100
                
                # Calculate regime-specific performance
                regime_returns = portfolio_df.loc[regime_mask, 'value'].pct_change().dropna()
                regime_sharpe = (regime_returns.mean() * 252) / (regime_returns.std() * np.sqrt(252)) if len(regime_returns) > 0 else 0
                
                print(f"\nRegime {regime} ({regime_name}):")
                print(f"  Days:         {n_days} ({pct:.1f}%)")
                print(f"  Sharpe Ratio: {regime_sharpe:.3f}")

        # ====================================================================
        # VISUALIZATION
        # ====================================================================
        
        if benchmark_df is not None:
            plot_regime_portfolio_with_benchmark(portfolio_history, benchmark_df)
        else:
            print("\n✗ Skipping benchmark visualization (no benchmark data)")

        print(f"\n{'='*80}")
        print("✓ EXECUTION COMPLETE (NO LOOK-AHEAD BIAS)")
        print(f"{'='*80}")

        return portfolio_history, trades, backtest.regime_history

    except Exception as e:
        print(f"\nERROR: {str(e)}")
        import traceback
        traceback.print_exc()
        return None, None, None


if __name__ == "__main__":
    portfolio_history, trades, regime_history = main()
